In [1]:
import numpy as np
import bgshr
import pandas
import matplotlib.pyplot as plt

In [17]:
cons_models = [
    "merged_cds_regulatory",
    "split_cds_regulatory",
    "merged_cds_phastcons",
    "split_cds_phastcons"]
mut_models = [
    "carlson",
    "gnomad",
    "roulette"]
chroms = list(range(1, 23))

df = pandas.read_csv("../models/equilibrium_granular_Ne/Ne_tbl.csv")
avg_df = pandas.read_csv("../models/equilibrium_single_Ne/avg_Ne_tbl.csv")
L_df = pandas.read_csv("../data/h38.genome")
Ls = [next(iter(L_df[L_df["Chrom"] == c]["Length"])) for c in chroms]

colors = ["tab:blue", "tab:orange", "tab:green"]

fig, axs = plt.subplots(2, 2, figsize=(12, 10), layout="constrained", sharex=True, sharey="row")

for ax, cons_model in zip(axs.flat, cons_models):
    objs = []
    for color, mut_model in zip(colors, mut_models):
        sub_df = df[
            (df["cons_model"] == cons_model)
            & (df["mut_model"] == mut_model)]
        Nes = [next(iter(sub_df[sub_df["chrom"] == c]["Ne"])) for c in chroms]
        for c, Ne, L in zip(chroms, Nes, Ls):
            ax.scatter(L, Ne, color="none")
            ax.text(L, Ne, c, va="center", ha="center", fontsize=9, color=color)

        avg_Ne = next(iter(avg_df[
            (avg_df["cons_model"] == cons_model)
            & (avg_df["mut_model"] == mut_model)]["avg_Ne"]))
        ax.hlines(avg_Ne, 0, 3e8, color=color, lw=1, alpha=0.6)
        x, *_ = ax.plot([], [], color=color)
        objs.append(x)
        
axs[0, 0].set_title("merged CDS + regulatory")
axs[0, 1].set_title("split CDS + regulatory")
axs[1, 0].set_title("merged CDS + phastcons")
axs[1, 1].set_title("split CDS + phastcons")

axs[0, 0].set_ylabel("Inferred $N_e$")
axs[1, 0].set_ylabel("Inferred $N_e$")

axs[0, 0].set_xlim(2.5e7, 2.75e8)
xticks = np.arange(5e7, 2.75e8, 2.5e7)
axs[0, 0].set_xticks(xticks, (xticks / 1e6).astype(int))
axs[1, 0].set_xlabel("Length (Mb)")
axs[1, 1].set_xlabel("Length (Mb)")

fig.legend(objs, ["Carlson", "Gnomad", "Roulette"], framealpha=0, loc="lower center", ncols=3, bbox_to_anchor=(0.5, -0.05))
plt.savefig(f"../figures/Ne_distributions.pdf", bbox_inches='tight')
plt.close()